# Solemne de Optimizacion: evidencia computacional con Tabu Search

Este cuaderno contiene la implementacion reproducible utilizada para respaldar el
informe `Solemne_Optimizacion_Respuesta.pdf`. La formulacion matematica se
mantiene en el informe principal y la ejecucion computacional se desarrolla en
Python mediante Tabu Search.

## Nota metodologica

Aunque el enunciado original considera implementacion en AMPL, en esta entrega se
utilizo un enfoque metaheuristico mediante Tabu Search implementado en Python. La
formulacion matematica se mantiene en el informe, mientras que la ejecucion
computacional y la validacion de resultados se presentan en el notebook adjunto.


In [ ]:
import pandas as pd
from IPython.display import Image, display

from figuras import generar_figuras
from modelo import (
    CAMIONES,
    COSTO_KM,
    DELTA,
    DEMANDA,
    DIST,
    ESCENARIOS,
    ESTACIONES,
    OPERACIONES_CARGA,
    PENAL_SHORTAGE,
    SERVICIO,
    TASA,
    VENTANA,
    cargas_iniciales,
    demanda_total_por_escenario,
    distancia_ruta,
    evaluar,
    hm,
    optimo_fuerza_bruta,
    perfil_fill_ratios,
    scheduling_makespan,
    tiempos_llegada,
    todas_las_soluciones,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

PARAM_TABU = {
    "iteraciones": 80,
    "tenencia": 7,
    "reinicios": 4,
    "semilla": 42,
    "penalizacion_inf": 100000,
}

artefactos = generar_figuras(
    iteraciones=PARAM_TABU["iteraciones"],
    tenencia=PARAM_TABU["tenencia"],
    reinicios=PARAM_TABU["reinicios"],
    semilla=PARAM_TABU["semilla"],
)
mejor_sol = artefactos["mejor"]
mejor_ev = artefactos["evaluacion"]
historial = artefactos["historial"]
meta_tabu = artefactos["meta"]

sol_operador = {
    "T1": {"ruta": [1, 3], "comp": {"C0": "R", "C1": "D"}},
    "T2": {"ruta": [2, 4], "comp": {"C0": "D", "C1": "R"}},
}
ev_operador = evaluar(sol_operador)


def fmt_hora(minutos):
    return f"{int(minutos // 60):02d}:{int(minutos % 60):02d}"


def ruta_texto(ruta):
    return "D->" + "->".join(map(str, ruta)) + "->D" if ruta else "No se usa"


def resumen_capacidad(sol):
    filas = []
    for camion in ("T1", "T2"):
        cargas, shortage = cargas_iniciales(camion, sol[camion]["ruta"], sol[camion]["comp"])
        for comp in ("C0", "C1"):
            fuel = sol[camion]["comp"][comp]
            requerido = sum(DEMANDA[j][fuel] for j in sol[camion]["ruta"])
            capacidad = CAMIONES[camion][comp]
            filas.append(
                {
                    "Camion": camion,
                    "Compartimento": comp,
                    "Producto": fuel,
                    "Demanda asignada (L)": requerido,
                    "Capacidad (L)": capacidad,
                    "Carga inicial (L)": cargas[comp],
                    "Shortage (L)": shortage[comp],
                    "Fill inicial": round(cargas[comp] / capacidad, 4),
                    "Cumple capacidad": requerido <= capacidad,
                }
            )
    return pd.DataFrame(filas)


def resumen_estabilidad(camion, ruta, comp):
    filas = []
    for etapa, fr_c0, fr_c1 in perfil_fill_ratios(camion, ruta, comp):
        filas.append(
            {
                "Camion": camion,
                "Etapa": etapa,
                "Fill C0": round(fr_c0, 4),
                "Fill C1": round(fr_c1, 4),
                "Diferencia": round(abs(fr_c0 - fr_c1), 4),
                "Cumple Delta": abs(fr_c0 - fr_c1) <= DELTA + 1e-9,
            }
        )
    return pd.DataFrame(filas)


def resumen_ventanas(sol):
    filas = []
    for camion in ("T1", "T2"):
        for estacion, llegada, inicio_servicio in tiempos_llegada(camion, sol[camion]["ruta"]):
            apertura, cierre = VENTANA[estacion]
            filas.append(
                {
                    "Camion": camion,
                    "Estacion": estacion,
                    "Llegada": fmt_hora(llegada),
                    "Inicio servicio": fmt_hora(inicio_servicio),
                    "Ventana": f"{fmt_hora(apertura)}-{fmt_hora(cierre)}",
                    "Espera (min)": inicio_servicio - llegada,
                    "Cumple": llegada <= cierre,
                }
            )
    return pd.DataFrame(filas)


def costo_desagregado(nombre, sol, ev):
    return {
        "Solucion": nombre,
        "Ruta T1": ruta_texto(sol["T1"]["ruta"]),
        "Ruta T2": ruta_texto(sol["T2"]["ruta"]),
        "Distancia total (km)": ev["dist_km"],
        "Costo distancia": ev["costo_distancia"],
        "Costo fijo": ev["costo_fijo"],
        "Shortage (L)": ev["shortage"],
        "Penalizacion shortage": ev["shortage"] * PENAL_SHORTAGE,
        "Costo total": ev["costo"],
        "Factible": ev["factible"],
    }


## 1. Datos del escenario base

Los parametros reproducidos a continuacion corresponden al escenario determinista
base utilizado para la evaluacion. Se consideran dos camiones con doble
compartimento, ventanas de tiempo por estacion, costos de distancia, costos fijos
por uso de camion y penalizacion por faltantes.


In [ ]:
tabla_estaciones = pd.DataFrame(
    [
        {
            "Estacion": j,
            "Regular (L)": DEMANDA[j]["R"],
            "Diesel (L)": DEMANDA[j]["D"],
            "Apertura": fmt_hora(VENTANA[j][0]),
            "Cierre": fmt_hora(VENTANA[j][1]),
        }
        for j in ESTACIONES
    ]
)

tabla_camiones = pd.DataFrame(
    [
        {
            "Camion": camion,
            "Capacidad C0 (L)": datos["C0"],
            "Capacidad C1 (L)": datos["C1"],
            "Costo fijo": datos["fijo"],
            "Salida programada": fmt_hora(datos["salida"]),
        }
        for camion, datos in CAMIONES.items()
    ]
)

tabla_distancias = pd.DataFrame(
    [[DIST[(i, j)] for j in [0] + ESTACIONES] for i in [0] + ESTACIONES],
    index=["D"] + ESTACIONES,
    columns=["D"] + ESTACIONES,
)

display(tabla_estaciones)
display(tabla_camiones)
display(tabla_distancias)

print(
    f"Costo por kilometro = ${COSTO_KM} | Penalizacion por shortage = ${PENAL_SHORTAGE}/L | "
    f"Servicio por estacion = {SERVICIO} min | Delta de estabilidad = {DELTA:.2f}"
)


## 2. Parametrizacion de la busqueda y criterio de evaluacion

La solucion se representa mediante una ruta ordenada por camion y una asignacion
de producto a cada compartimento. La funcion de evaluacion considera:

- costo de distancia,
- costo fijo por camion utilizado,
- penalizacion por shortage,
- validacion de capacidad por compartimento,
- validacion de estabilidad de carga con `Delta = 0.30`,
- validacion de ventanas de tiempo permitiendo espera cuando el camion arriba
  antes de la apertura.

La busqueda Tabu explora vecinos por movimientos de traspaso, intercambio,
reordenamiento de ruta y cambio de asignacion de compartimentos. Se utiliza lista
tabu con criterio de aspiracion y reinicios reproducibles.


In [ ]:
tabla_parametros_tabu = pd.DataFrame(
    [
        {"Parametro": "Iteraciones por arranque", "Valor": PARAM_TABU["iteraciones"]},
        {"Parametro": "Tenure tabu", "Valor": PARAM_TABU["tenencia"]},
        {"Parametro": "Reinicios adicionales", "Valor": PARAM_TABU["reinicios"]},
        {"Parametro": "Semilla", "Valor": PARAM_TABU["semilla"]},
        {"Parametro": "Penalizacion por infactibilidad", "Valor": PARAM_TABU["penalizacion_inf"]},
    ]
)

tabla_arranques = pd.DataFrame(meta_tabu["arranques"])
display(tabla_parametros_tabu)
display(tabla_arranques)


## 3. Evaluacion de la solucion propuesta por el operador

Se analiza la propuesta del operador en terminos de capacidad, estabilidad,
ventanas de tiempo y costo total. La evaluacion confirma que el costo de distancia
declarado no coincide con el valor real y que la ruta del camion T2 viola la
restriccion de estabilidad luego de atender la estacion 2.


In [ ]:
display(resumen_capacidad(sol_operador))

estabilidad_operador = pd.concat(
    [
        resumen_estabilidad("T1", sol_operador["T1"]["ruta"], sol_operador["T1"]["comp"]),
        resumen_estabilidad("T2", sol_operador["T2"]["ruta"], sol_operador["T2"]["comp"]),
    ],
    ignore_index=True,
)
display(estabilidad_operador)
display(resumen_ventanas(sol_operador))

tabla_costos_operador = pd.DataFrame(
    [
        {"Concepto": "Distancia T1", "Valor": distancia_ruta(sol_operador["T1"]["ruta"])},
        {"Concepto": "Distancia T2", "Valor": distancia_ruta(sol_operador["T2"]["ruta"])},
        {"Concepto": "Distancia total", "Valor": ev_operador["dist_km"]},
        {"Concepto": "Costo distancia real", "Valor": ev_operador["costo_distancia"]},
        {"Concepto": "Costo fijo", "Valor": ev_operador["costo_fijo"]},
        {"Concepto": "Shortage", "Valor": ev_operador["shortage"]},
        {"Concepto": "Costo total real", "Valor": ev_operador["costo"]},
        {"Concepto": "Costo distancia declarado por el operador", "Valor": 260},
    ]
)
display(tabla_costos_operador)

display(Image(filename=artefactos["archivos"]["rutas_operador"]))
display(Image(filename=artefactos["archivos"]["estabilidad_t2"]))


## 4. Propuesta de mejora mediante Tabu Search

La mejor solucion encontrada por Tabu Search asigna las estaciones 1, 2 y 4 al
camion T1 y deja la estacion 3 al camion T2. El resultado mejora el costo total,
elimina la violacion de estabilidad observada en la propuesta del operador y
mantiene factibilidad en capacidad y ventanas de tiempo.


In [ ]:
tabla_mejor_solucion = pd.DataFrame(
    [
        {
            "Camion": camion,
            "Ruta": ruta_texto(mejor_sol[camion]["ruta"]),
            "Compartimento C0": mejor_sol[camion]["comp"]["C0"],
            "Compartimento C1": mejor_sol[camion]["comp"]["C1"],
            "Distancia (km)": distancia_ruta(mejor_sol[camion]["ruta"]),
        }
        for camion in ("T1", "T2")
    ]
)

comparacion = pd.DataFrame(
    [
        costo_desagregado("Operador", sol_operador, ev_operador),
        costo_desagregado("Tabu Search", mejor_sol, mejor_ev),
    ]
)

estabilidad_mejor = pd.concat(
    [
        resumen_estabilidad("T1", mejor_sol["T1"]["ruta"], mejor_sol["T1"]["comp"]),
        resumen_estabilidad("T2", mejor_sol["T2"]["ruta"], mejor_sol["T2"]["comp"]),
    ],
    ignore_index=True,
)

display(tabla_mejor_solucion)
display(comparacion)
display(estabilidad_mejor)
display(resumen_ventanas(mejor_sol))

display(Image(filename=artefactos["archivos"]["convergencia"]))
display(Image(filename=artefactos["archivos"]["rutas_optimo"]))


## 5. Validacion exhaustiva por enumeracion

Dado el tamano acotado de la instancia, fue posible enumerar todas las
asignaciones de estaciones, los ordenes de visita y las asignaciones de
compartimentos. Esta verificacion exhaustiva permite confirmar que la solucion
encontrada por Tabu Search coincide con el mejor costo factible del espacio
completo de soluciones bajo los supuestos implementados en el notebook.


In [ ]:
registros = []
for sol in todas_las_soluciones():
    ev = evaluar(sol)
    registros.append(
        {
            "Ruta T1": ruta_texto(sol["T1"]["ruta"]),
            "Ruta T2": ruta_texto(sol["T2"]["ruta"]),
            "Comp T1": f"C0={sol['T1']['comp']['C0']} / C1={sol['T1']['comp']['C1']}",
            "Comp T2": f"C0={sol['T2']['comp']['C0']} / C1={sol['T2']['comp']['C1']}",
            "Distancia (km)": ev["dist_km"],
            "Costo total": ev["costo"],
            "Factible": ev["factible"],
        }
    )

enumeracion = pd.DataFrame(registros)
factibles = enumeracion[enumeracion["Factible"]].sort_values(
    ["Costo total", "Distancia (km)", "Ruta T1", "Ruta T2"]
).reset_index(drop=True)
mejor_bruta, eval_bruta, n_factibles = optimo_fuerza_bruta()

resumen_enumeracion = pd.DataFrame(
    [
        {"Metrica": "Soluciones enumeradas", "Valor": len(enumeracion)},
        {"Metrica": "Soluciones factibles", "Valor": n_factibles},
        {"Metrica": "Mejor costo por fuerza bruta", "Valor": eval_bruta["costo"]},
        {"Metrica": "Mejor costo por Tabu Search", "Valor": mejor_ev["costo"]},
        {
            "Metrica": "Coincidencia Tabu Search vs fuerza bruta",
            "Valor": mejor_ev["costo"] == eval_bruta["costo"],
        },
    ]
)

display(resumen_enumeracion)
display(factibles.head(10))
print("Ruta optima por fuerza bruta T1:", ruta_texto(mejor_bruta["T1"]["ruta"]))
print("Ruta optima por fuerza bruta T2:", ruta_texto(mejor_bruta["T2"]["ruta"]))


## 6. Scheduling de carga en el deposito

El scheduling considera una sola bahia por combustible, limpieza al termino de
cada carga y precedencia interna dentro de cada camion. La solucion obtenida
minimiza el makespan y verifica que ambos camiones quedan listos antes de sus
horas programadas de salida.


In [ ]:
duraciones, inicios, terminos, listo, makespan = scheduling_makespan()

tabla_operaciones = pd.DataFrame(
    [
        {
            "Operacion": op_id,
            "Camion": camion,
            "Compartimento": compartimento,
            "Producto": fuel,
            "Litros": litros,
            "Tasa (L/min)": TASA[fuel],
            "Duracion total (min)": duraciones[op_id],
            "Inicio (min)": inicios[op_id],
            "Termino (min)": terminos[op_id],
        }
        for op_id, camion, compartimento, fuel, litros in OPERACIONES_CARGA
    ]
)

tabla_salidas = pd.DataFrame(
    [
        {
            "Camion": camion,
            "Listo si la carga inicia a las 04:00": fmt_hora(hm(4, 0) + listo[camion]),
            "Salida programada": fmt_hora(CAMIONES[camion]["salida"]),
            "Cumple salida": hm(4, 0) + listo[camion] <= CAMIONES[camion]["salida"],
        }
        for camion in ("T1", "T2")
    ]
)

display(tabla_operaciones)
display(tabla_salidas)
print(f"Makespan optimo = {makespan:.1f} minutos")
display(Image(filename=artefactos["archivos"]["gantt"]))


## 7. Extension estocastica de dos etapas

La extension estocastica conserva las decisiones de primera etapa asociadas a la
ruta y al uso de camiones, y desplaza a segunda etapa las decisiones de carga y
shortage dependientes del escenario. La siguiente tabla resume las demandas
totales por escenario y la separacion conceptual entre ambas etapas.


In [ ]:
tabla_escenarios = pd.DataFrame(
    [
        {
            "Escenario": s,
            "Probabilidad": datos["prob"],
            "Regular total (L)": sum(datos[j]["R"] for j in ESTACIONES),
            "Diesel total (L)": sum(datos[j]["D"] for j in ESTACIONES),
            "Demanda total (L)": sum(datos[j]["R"] + datos[j]["D"] for j in ESTACIONES),
        }
        for s, datos in ESCENARIOS.items()
    ]
)

tabla_etapas = pd.DataFrame(
    [
        {
            "Etapa": "Primera etapa",
            "Decision principal": "Rutas, uso de camiones y asignacion producto-compartimento",
            "Momento": "Antes de observar la demanda",
        },
        {
            "Etapa": "Segunda etapa",
            "Decision principal": "Cargas efectivas y shortage por escenario",
            "Momento": "Despues de observar la demanda del escenario",
        },
    ]
)

display(tabla_escenarios)
display(tabla_etapas)

resumen_capacidad = pd.DataFrame(
    [
        {"Concepto": "Capacidad total flota", "Valor": sum(c["C0"] + c["C1"] for c in CAMIONES.values())},
        {"Concepto": "Capacidad maxima Regular", "Valor": CAMIONES["T1"]["C0"] + CAMIONES["T2"]["C1"]},
        {"Concepto": "Capacidad maxima Diesel", "Valor": CAMIONES["T1"]["C1"] + CAMIONES["T2"]["C0"]},
    ]
)
display(resumen_capacidad)


## 8. Sintesis reproducible

La ejecucion anterior reproduce los resultados centrales que se reportan en el
PDF:

- la propuesta del operador no es factible por estabilidad y su costo real es
  `$1200`,
- la mejor solucion factible encontrada es `T1: D->1->2->4->D` y `T2: D->3->D`,
- el costo total mejorado es `$1130`,
- el scheduling de carga tiene makespan optimo de `32.5` minutos,
- la enumeracion exhaustiva confirma que el costo de Tabu Search coincide con el
  optimo global del espacio factible implementado.

Con ello, el notebook deja trazabilidad completa de tablas, verificaciones y
figuras empleadas en la entrega final.
